In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
%cd drive/MyDrive/

In [ ]:
!rm -rf FPL_forecast
!git clone https://github.com/bragehs/FPL_forecast.git

In [ ]:
%cd FPL_forecast/predictor/

In [1]:
file_path = '/content/drive/MyDrive/colab_fpl'
file_path

'/content/drive/MyDrive/colab_fpl'

In [1]:
import os
import torch
from training import train_model, hyperparameter_tuning
from model import FPLComponentModel

In [2]:
file_path = os.getcwd() + "/processed_data"
file_path

'/Users/bragehs/Documents/FPL_forecast/predictor/processed_data'

In [3]:
X_train_numeric = torch.load(file_path + "/X_train.pt", weights_only=True)
X_train_static = torch.load(file_path + "/X_static_train.pt", weights_only=True)
y_train = torch.load(file_path + "/y_train.pt", weights_only=True)

X_val_numeric = torch.load(file_path + "/X_val.pt", weights_only=True)
X_val_static = torch.load(file_path + "/X_static_val.pt", weights_only=True)
y_val = torch.load(file_path + "/y_val.pt", weights_only=True)

print(f"Train sequences: {X_train_numeric.shape}, {X_train_static.shape}, Targets: {y_train.shape}")
print(f"Validation sequences: {X_val_numeric.shape}, {X_val_static.shape}, Targets: {y_val.shape}")

Train sequences: torch.Size([56230, 5, 19]), torch.Size([56230, 21]), Targets: torch.Size([56230, 1, 5])
Validation sequences: torch.Size([27283, 5, 19]), torch.Size([27283, 21]), Targets: torch.Size([27283, 1])


In [4]:
# Hyperparameter tuning
best_params = hyperparameter_tuning(X_train_numeric=X_train_numeric, y_train=y_train,X_train_static=X_train_static,
                                    X_val_numeric=X_val_numeric, X_val_static=X_val_static, y_val=y_val,
                                    epochs=1, n_trials=1, num_workers=4,
                                    )

Running random search with 1 trials...

Trial 1/1
Params: {'learning_rate': 0.001, 'hidden_dim': 96, 'weight_decay': 0.001, 'lstm_layers': 4, 'dropout': 0.0, 'batch_size': 64, 'weight': {'xg': 1.0, 'xa': 1.0, 'cs': 0.5, 'mins': 0.5}}
New best RMSE: 2.0437

Top 5 hyperparameter combinations:
1. RMSE: 2.0437, Params: {'learning_rate': 0.001, 'hidden_dim': 96, 'weight_decay': 0.001, 'lstm_layers': 4, 'dropout': 0.0, 'batch_size': 64, 'weight': {'xg': 1.0, 'xa': 1.0, 'cs': 0.5, 'mins': 0.5}, 'rmse': 2.0436602315945085}

Best hyperparameters: {'learning_rate': 0.001, 'hidden_dim': 96, 'weight_decay': 0.001, 'lstm_layers': 4, 'dropout': 0.0, 'batch_size': 64, 'weight': {'xg': 1.0, 'xa': 1.0, 'cs': 0.5, 'mins': 0.5}}
Best RMSE: 2.0437


In [ ]:
model = FPLComponentModel(
        numeric_seq_dim=X_train_numeric.shape[-1],
        static_dim=X_train_static.shape[-1],
        hidden_dim=best_params["hidden_dim"],
        lstm_layers=best_params["lstm_layers"],
        dropout=best_params["dropout"],
    )


Training final model with best hyperparameters...


In [6]:
# Full training with best hyperparameters
train_model(
    model=model,
    X_train_numeric=X_train_numeric, X_train_static=X_train_static, y_train=y_train,
    X_val_numeric=X_val_numeric, X_val_static=X_val_static, y_val=y_val,
    learning_rate=best_params["learning_rate"],
    weight_decay=best_params["weight_decay"],
    batch_size=best_params["batch_size"],
    weights=best_params["weight"],
    epochs=1,
    verbose=2,
    num_workers=4,
        )

Epoch  1/1: 100%|██████████| 879/879 [01:04<00:00, 13.55it/s, loss=0.5327, lr=0.001]


Epoch 1 Training MSE: 0.5270


Epoch 1 validation RMSE: 2.0048
Epoch 1 validation MAE: 1.0303
Best model saved at epoch 1 with MAE: 2.0048


2.004767003394918